In [0]:
%pip install xgboost

In [0]:
%pip install tensorflow

In [0]:
# Importing Libraries

import os
import numpy as np
import pandas as pd
import xgboost as xgb
import mlflow
from pyspark.sql.functions import col
from pyspark.ml.functions import vector_to_array
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.metrics import (
    roc_auc_score, average_precision_score, precision_recall_curve
)
from tensorflow import keras

os.environ['MLFLOW_DFS_TMP'] = '/Volumes/workspace/ml_layer/mlflow_tmp'

def ensure_catalog():
    spark.sql("USE CATALOG workspace")
    spark.sql("USE DATABASE ml_layer")

ensure_catalog()

username = spark.sql("SELECT current_user()").collect()[0][0]
mlflow.set_experiment(f"/Users/{username}/combined_scoring")

# Loading data with original transaction amounts for business impact

def load_with_amounts(train_table, test_table):
    """Load test set with original amount column for business impact calculation."""
    train_df = spark.table(train_table) \
                    .withColumn("is_fraud", col("is_fraud").cast("double"))
    test_df  = spark.table(test_table) \
                    .withColumn("is_fraud", col("is_fraud").cast("double"))

    def extract(df):
        arr = df.withColumn("features_arr", vector_to_array("features"))
        pdf = arr.select("features_arr", "is_fraud").toPandas()
        X   = np.array(pdf["features_arr"].tolist())
        y   = pdf["is_fraud"].values
        # log_amount is feature 0 in transactions schema — reverse log1p
        amounts = np.expm1(X[:, 0])
        return X, y, amounts

    X_train, y_train, _              = extract(train_df)
    X_test,  y_test,  amounts_test   = extract(test_df)
    return X_train, y_train, X_test, y_test, amounts_test


print("Loading transactions data...")
X_txn_train, y_txn_train, X_txn_test, y_txn_test, amounts = load_with_amounts(
    "workspace.ml_layer.transaction_train_features",
    "workspace.ml_layer.transaction_test_features"
)

# Normalization for autoencoder
scaler = StandardScaler()
scaler.fit(X_txn_train[y_txn_train == 0])
X_txn_train_sc = scaler.transform(X_txn_train)
X_txn_test_sc  = scaler.transform(X_txn_test)

# Training production XGBoost using best Optuna params

fraud_count      = int(y_txn_train.sum())
legit_count      = len(y_txn_train) - fraud_count
scale_pos_weight = legit_count / fraud_count

xgb_model = xgb.XGBClassifier(
    n_estimators          = 400,
    max_depth             = 5,
    learning_rate         = 0.03,
    subsample             = 0.75,
    colsample_bytree      = 0.7,
    colsample_bylevel     = 0.8,
    min_child_weight      = 20,
    gamma                 = 0.1,
    reg_alpha             = 0.05,
    reg_lambda            = 1.5,
    scale_pos_weight      = scale_pos_weight,
    eval_metric           = "aucpr",
    early_stopping_rounds = 40,
    random_state          = 42,
    n_jobs                = -1
)
xgb_model.fit(X_txn_train, y_txn_train,
              eval_set=[(X_txn_test, y_txn_test)],
              verbose=False)

xgb_scores = xgb_model.predict_proba(X_txn_test)[:, 1]

# Training Production Autoencoder

def build_autoencoder(n_features):
    encoding_dim = max(2, int(round(n_features * 0.3)))
    hidden_1     = max(encoding_dim + 2, int(round(n_features * 0.8)))
    hidden_2     = max(encoding_dim + 1, int(round(n_features * 0.5)))

    inputs = keras.Input(shape=(n_features,))
    x = keras.layers.Dense(hidden_1, activation="relu")(inputs)
    x = keras.layers.BatchNormalization()(x)
    x = keras.layers.Dropout(0.2)(x)
    x = keras.layers.Dense(hidden_2, activation="relu")(x)
    x = keras.layers.BatchNormalization()(x)
    encoded = keras.layers.Dense(encoding_dim, activation="relu")(x)

    x = keras.layers.Dense(hidden_2, activation="relu")(encoded)
    x = keras.layers.BatchNormalization()(x)
    x = keras.layers.Dense(hidden_1, activation="relu")(x)
    decoded = keras.layers.Dense(n_features, activation="linear")(x)

    model = keras.Model(inputs, decoded)
    model.compile(optimizer=keras.optimizers.Adam(0.001), loss="mse")
    return model

X_legit         = X_txn_train_sc[y_txn_train == 0]
val_size        = int(0.1 * len(X_legit))
ae              = build_autoencoder(X_legit.shape[1])
ae.fit(
    X_legit[val_size:], X_legit[val_size:],
    epochs=20, batch_size=512,
    validation_data=(X_legit[:val_size], X_legit[:val_size]),
    callbacks=[keras.callbacks.EarlyStopping(patience=5, restore_best_weights=True)],
    verbose=0
)

reconstructed = ae.predict(X_txn_test_sc, verbose=0)
ae_scores     = np.mean((X_txn_test_sc - reconstructed) ** 2, axis=1)

# Combined Scored

score_scaler = MinMaxScaler()
xgb_norm     = score_scaler.fit_transform(xgb_scores.reshape(-1, 1)).flatten()
ae_norm      = score_scaler.fit_transform(ae_scores.reshape(-1, 1)).flatten()

WEIGHT_XGB = 0.75   # primary scorer — best PR-AUC
WEIGHT_AE  = 0.25   # secondary signal — catches novel patterns

combined_scores = WEIGHT_XGB * xgb_norm + WEIGHT_AE * ae_norm

# Complementary Analysis

xgb_threshold = 0.8558   # XGBoost optimal threshold from notebook 01
ae_threshold  = np.percentile(ae_scores, 95)

xgb_caught = (xgb_scores >= xgb_threshold) & (y_txn_test == 1)
ae_caught  = (ae_scores >= ae_threshold)   & (y_txn_test == 1)

both          = (xgb_caught & ae_caught).sum()
only_xgb      = (xgb_caught & ~ae_caught).sum()
only_ae       = (~xgb_caught & ae_caught).sum()
neither       = (~xgb_caught & ~ae_caught & (y_txn_test == 1)).sum()
total_fraud   = int(y_txn_test.sum())

print("\n" + "="*60)
print("  COMPLEMENTARITY ANALYSIS — XGBoost vs Autoencoder")
print("="*60)
print(f"  Total fraud cases:    {total_fraud:,}")
print(f"  Caught by both:       {both:,} ({100*both/total_fraud:.1f}%)")
print(f"  Only XGBoost caught:  {only_xgb:,} ({100*only_xgb/total_fraud:.1f}%)")
print(f"  Only Autoencoder:     {only_ae:,} ({100*only_ae/total_fraud:.1f}%) ← AE unique value")
print(f"  Missed by both:       {neither:,} ({100*neither/total_fraud:.1f}%)")

ae_unique_pct = 100 * only_ae / total_fraud
if ae_unique_pct >= 5:
    ae_recommendation = f"✅ KEEP AE — adds {ae_unique_pct:.1f}% unique fraud coverage"
elif ae_unique_pct >= 2:
    ae_recommendation = f"⚠️  AE in shadow mode — {ae_unique_pct:.1f}% marginal value"
else:
    ae_recommendation = f"❌ DROP AE — only {ae_unique_pct:.1f}% unique fraud catch"

print(f"\n  Recommendation: {ae_recommendation}")

# Risk Tier Assessment

def assign_risk_tier(score):
    if score >= 0.80:
        return "CRITICAL"
    elif score >= 0.50:
        return "HIGH"
    elif score >= 0.20:
        return "MEDIUM"
    else:
        return "LOW"

risk_tiers = pd.Series([assign_risk_tier(s) for s in combined_scores])

action_map = {
    "CRITICAL": "AUTO-BLOCK — freeze immediately, alert customer",
    "HIGH"    : "MANUAL REVIEW — fraud analyst reviews within 1 hour",
    "MEDIUM"  : "ENHANCED MONITORING — flag for next-day batch review",
    "LOW"     : "APPROVE — proceed with standard processing"
}

# Business Impact qualification

total_fraud_value     = amounts[y_txn_test == 1].sum()
total_legit_value     = amounts[y_txn_test == 0].sum()
fraud_caught_combined = (combined_scores >= 0.5) & (y_txn_test == 1)
fraud_value_caught    = amounts[fraud_caught_combined].sum()
fraud_value_missed    = total_fraud_value - fraud_value_caught

# False positive cost estimation
# Industry standard: blocked legit transaction costs ~$15 in customer friction
FP_COST_PER_BLOCK = 15
false_positives   = ((combined_scores >= 0.5) & (y_txn_test == 0)).sum()
fp_cost_total     = false_positives * FP_COST_PER_BLOCK

# Recovery rate: typically only 10-20% of caught fraud is fully recovered
RECOVERY_RATE = 0.15
recovered_value = fraud_value_caught * RECOVERY_RATE

print("\n" + "="*60)
print("  BUSINESS IMPACT — ANNUALIZED PROJECTION")
print("="*60)
print(f"  Test set fraud value         : ${total_fraud_value:>15,.2f}")
print(f"  Fraud value detected         : ${fraud_value_caught:>15,.2f} ({100*fraud_value_caught/total_fraud_value:.1f}%)")
print(f"  Fraud value missed           : ${fraud_value_missed:>15,.2f}")
print(f"  Recovered value (15% rate)   : ${recovered_value:>15,.2f}")
print(f"  False positive cost          : ${fp_cost_total:>15,.2f} ({false_positives:,} blocks)")
print(f"  Net business value           : ${recovered_value - fp_cost_total:>15,.2f}")

# Annualize: test set covers ~3 months of transactions
ANNUAL_MULTIPLIER = 4
annual_recovered  = recovered_value * ANNUAL_MULTIPLIER
annual_fp_cost    = fp_cost_total * ANNUAL_MULTIPLIER
annual_net        = annual_recovered - annual_fp_cost

print(f"\n  ANNUAL PROJECTION:")
print(f"     Recovered fraud value     : ${annual_recovered:>15,.2f}")
print(f"     Customer friction cost    : ${annual_fp_cost:>15,.2f}")
print(f"     NET ANNUAL BENEFIT        : ${annual_net:>15,.2f}")

# Saving for dashboards

production_results = pd.DataFrame({
    "transaction_idx" : range(len(combined_scores)),
    "xgb_score"       : xgb_scores,
    "ae_score"        : ae_scores,
    "combined_score"  : combined_scores,
    "risk_tier"       : risk_tiers,
    "recommended_action": [action_map[t] for t in risk_tiers],
    "actual_fraud"    : y_txn_test,
    "transaction_amount": amounts
})

# Risk tier summary for dashboard
tier_summary = production_results.groupby("risk_tier").agg(
    total_count      = ("actual_fraud", "count"),
    fraud_caught     = ("actual_fraud", "sum"),
    avg_combined_score = ("combined_score", "mean"),
    total_amount     = ("transaction_amount", "sum")
).reset_index()
tier_summary["fraud_rate_pct"] = (tier_summary["fraud_caught"] / tier_summary["total_count"] * 100).round(2)

# Business impact summary
business_impact = pd.DataFrame([{
    "test_period_fraud_value"     : float(total_fraud_value),
    "fraud_value_caught"          : float(fraud_value_caught),
    "fraud_value_missed"          : float(fraud_value_missed),
    "detection_rate_by_value"     : float(fraud_value_caught / total_fraud_value),
    "false_positives_count"       : int(false_positives),
    "false_positive_cost"         : float(fp_cost_total),
    "annual_recovered_projection" : float(annual_recovered),
    "annual_fp_cost_projection"   : float(annual_fp_cost),
    "annual_net_benefit"          : float(annual_net),
    "ae_unique_fraud_pct"         : float(ae_unique_pct),
    "ae_production_recommendation": ae_recommendation
}])

# Save all to Delta
spark.createDataFrame(tier_summary).write.format("delta") \
    .mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable("workspace.ml_layer.production_risk_tiers")

spark.createDataFrame(business_impact).write.format("delta") \
    .mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable("workspace.ml_layer.production_business_impact")

print("\nProduction scoring complete — results saved to Delta")